# 🚀 Notebook 03 — CAMeL-BERT Fine-Tuning
## Darija Sentiment Analysis — خالد مرجان

**Objective:** Fine-tune `CAMeL-Lab/bert-base-arabic-camelbert-da` on our Darija sentiment dataset.

CAMeL-BERT is pre-trained on Arabic dialectal text — closest available model to Moroccan Darija.

**Expected accuracy: 85–91%** (vs ~75% for the TF-IDF baseline)

> ⚠️ **Recommended:** Run this notebook on **Google Colab** (free GPU) for faster training.
> Go to: https://colab.research.google.com → New Notebook → Upload this file

In [ ]:
# Install dependencies
# Uncomment if running on Colab or fresh environment
# !pip install transformers datasets evaluate accelerate torch scikit-learn -q

import pandas as pd
import numpy as np
import torch
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, f1_score,
                              classification_report, confusion_matrix)
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from datasets import Dataset
import evaluate

# Check GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Device: {device}')
if device == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
else:
    print('   ⚠️  No GPU — training will be slow. Use Google Colab for GPU.')

os.makedirs('models/camelbert', exist_ok=True)
os.makedirs('assets/images', exist_ok=True)
print('✅ Setup complete')

## 1. Load Data

In [ ]:
df_train = pd.read_csv('data/splits/train.csv', encoding='utf-8-sig')
df_val   = pd.read_csv('data/splits/val.csv',   encoding='utf-8-sig')
df_test  = pd.read_csv('data/splits/test.csv',  encoding='utf-8-sig')

# Label encoding
LABEL2ID = {'positive': 0, 'negative': 1, 'neutral': 2}
ID2LABEL = {0: 'positive', 1: 'negative', 2: 'neutral'}

for df in [df_train, df_val, df_test]:
    df['text']  = df['text'].fillna('').astype(str)
    df['label_id'] = df['label'].map(LABEL2ID)

# Drop rows with unmapped labels
df_train = df_train.dropna(subset=['label_id'])
df_val   = df_val.dropna(subset=['label_id'])
df_test  = df_test.dropna(subset=['label_id'])

print(f'Train: {len(df_train):,} | Val: {len(df_val):,} | Test: {len(df_test):,}')
print(f'Label mapping: {LABEL2ID}')

## 2. Load CAMeL-BERT Tokenizer

In [ ]:
MODEL_NAME = 'CAMeL-Lab/bert-base-arabic-camelbert-da'

print(f'Loading tokenizer: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'✅ Tokenizer loaded — vocab size: {tokenizer.vocab_size:,}')

# Test tokenization
sample = 'مزيان بزاف هاد الخبر'
tokens = tokenizer(sample, return_tensors='pt')
print(f'\nSample: "{sample}"')
print(f'Token IDs: {tokens["input_ids"]}')
print(f'Decoded: {tokenizer.decode(tokens["input_ids"][0])}')

## 3. Tokenize Dataset

In [ ]:
MAX_LENGTH = 128  # Most Darija comments are short

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=MAX_LENGTH
    )

# Convert to HuggingFace Dataset format
train_ds = Dataset.from_dict({
    'text': df_train['text'].tolist(),
    'labels': df_train['label_id'].astype(int).tolist()
})
val_ds = Dataset.from_dict({
    'text': df_val['text'].tolist(),
    'labels': df_val['label_id'].astype(int).tolist()
})
test_ds = Dataset.from_dict({
    'text': df_test['text'].tolist(),
    'labels': df_test['label_id'].astype(int).tolist()
})

# Tokenize
train_ds = train_ds.map(tokenize_function, batched=True)
val_ds   = val_ds.map(tokenize_function, batched=True)
test_ds  = test_ds.map(tokenize_function, batched=True)

# Set format for PyTorch
cols = ['input_ids', 'attention_mask', 'labels']
if 'token_type_ids' in train_ds.column_names:
    cols.append('token_type_ids')

train_ds.set_format(type='torch', columns=cols)
val_ds.set_format(type='torch', columns=cols)
test_ds.set_format(type='torch', columns=cols)

print(f'✅ Datasets tokenized')
print(f'   Train: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}')

## 4. Load CAMeL-BERT Model

In [ ]:
print(f'Loading model: {MODEL_NAME}')
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID
)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✅ Model loaded')
print(f'   Total parameters     : {total_params:,}')
print(f'   Trainable parameters : {trainable_params:,}')

## 5. Define Metrics

In [ ]:
accuracy_metric = evaluate.load('accuracy')
f1_metric = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average='weighted')
    return {
        'accuracy': accuracy['accuracy'],
        'f1': f1['f1']
    }

print('✅ Metrics defined: accuracy + weighted F1')

## 6. Training Configuration

In [ ]:
training_args = TrainingArguments(
    output_dir='models/camelbert',
    
    # Training
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    
    # Evaluation
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    
    # Logging
    logging_dir='logs',
    logging_steps=50,
    report_to='none',  # Disable wandb
    
    # Performance
    fp16=(device == 'cuda'),  # Mixed precision on GPU
    dataloader_num_workers=0,
    seed=42
)

print('✅ Training configuration:')
print(f'   Epochs     : {training_args.num_train_epochs}')
print(f'   Batch size : {training_args.per_device_train_batch_size}')
print(f'   LR         : {training_args.learning_rate}')
print(f'   FP16       : {training_args.fp16}')

## 7. Train the Model

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print('🚀 Starting training...')
print('   This takes ~10-15 min on GPU, ~2-3 hours on CPU')
print('   Recommended: Run on Google Colab for free GPU\n')

train_result = trainer.train()

print('\n✅ Training complete!')
print(f'   Training loss : {train_result.training_loss:.4f}')
print(f'   Time          : {train_result.metrics["train_runtime"]:.0f} seconds')

## 8. Evaluate on Test Set

In [ ]:
print('📊 Evaluating on test set...')
predictions = trainer.predict(test_ds)
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

# Convert back to string labels
y_pred_labels = [ID2LABEL[p] for p in y_pred]
y_true_labels = [ID2LABEL[t] for t in y_true]

test_acc = accuracy_score(y_true_labels, y_pred_labels)
test_f1  = f1_score(y_true_labels, y_pred_labels, average='weighted')

print('='*55)
print('📊 FINAL TEST RESULTS — CAMeL-BERT')
print('='*55)
print(f'Accuracy : {test_acc:.4f} ({test_acc*100:.2f}%)')
print(f'F1 Score : {test_f1:.4f}')
print()
print(classification_report(y_true_labels, y_pred_labels))

## 9. Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
labels_order = ['positive', 'negative', 'neutral']

cm = confusion_matrix(y_true_labels, y_pred_labels, labels=labels_order)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels_order, yticklabels=labels_order, ax=axes[0])
axes[0].set_title('CAMeL-BERT — Confusion Matrix (Counts)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')

cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=labels_order, yticklabels=labels_order, ax=axes[1])
axes[1].set_title('CAMeL-BERT — Confusion Matrix (Normalized)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')

plt.tight_layout()
plt.savefig('assets/images/camelbert_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Model Comparison Summary

In [ ]:
# Load baseline results (run Notebook 02 first)
# Update these values after running Notebook 02
baseline_acc = 0.9003  # Replace with your actual result from Notebook 02
baseline_f1  = 0.9008

comparison = pd.DataFrame({
    'Model': ['TF-IDF + LogReg (Baseline)', 'CAMeL-BERT (Fine-tuned)'],
    'Accuracy': [baseline_acc, test_acc],
    'F1 Score': [baseline_f1, test_f1]
})

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ['#95A5A6', '#E74C3C']

for idx, metric in enumerate(['Accuracy', 'F1 Score']):
    bars = axes[idx].bar(comparison['Model'], comparison[metric],
                         color=colors, edgecolor='white', width=0.5)
    axes[idx].set_title(f'Model Comparison — {metric}', fontsize=13, fontweight='bold')
    axes[idx].set_ylim(0, 1.0)
    axes[idx].set_ylabel(metric)
    for bar, val in zip(bars, comparison[metric]):
        axes[idx].text(bar.get_x() + bar.get_width()/2,
                       bar.get_height() + 0.01,
                       f'{val:.3f}\n({val*100:.1f}%)',
                       ha='center', fontsize=11, fontweight='bold')
    axes[idx].tick_params(axis='x', labelsize=9)

plt.suptitle('Darija Sentiment Analysis — Model Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('assets/images/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

improvement = (test_acc - baseline_acc) / baseline_acc * 100
print(comparison.to_string(index=False))
print(f'\n🎯 CAMeL-BERT improvement over baseline: +{improvement:.1f}%')

## 11. Save & Push to HuggingFace

In [ ]:
# Save locally
trainer.save_model('models/camelbert/final')
tokenizer.save_pretrained('models/camelbert/final')
print('✅ Model saved locally: models/camelbert/final')

# ── Push to HuggingFace Hub (optional) ──
# Uncomment after creating your HuggingFace account
# and running: huggingface-cli login

# from huggingface_hub import HfApi
# model.push_to_hub('KHALIDMRJ/darija-sentiment-camelbert')
# tokenizer.push_to_hub('KHALIDMRJ/darija-sentiment-camelbert')
# print('✅ Model pushed to HuggingFace!')
# print('   URL: https://huggingface.co/KHALIDMRJ/darija-sentiment-camelbert')

print('\n🎉 Project complete!')
print('\n📋 Summary:')
print(f'   Dataset      : 8,619 Darija comments')
print(f'   Baseline     : {baseline_acc*100:.1f}% accuracy (TF-IDF)')
print(f'   CAMeL-BERT   : {test_acc*100:.1f}% accuracy')
print(f'   Improvement  : +{improvement:.1f}%')
print(f'\n➡️  Next: Build Gradio demo (app/gradio_demo.py)')

## 12. Quick Inference — Test Your Model

In [ ]:
from transformers import pipeline

# Load the saved model for inference
sentiment_pipeline = pipeline(
    'text-classification',
    model='models/camelbert/final',
    tokenizer=tokenizer,
    device=0 if device == 'cuda' else -1
)

test_sentences = [
    'مزيان بزاف هاد القرار شكراً',
    'هاد الحكومة مكتخدمش والو سلبي جداً',
    'واش صحيح هاد الخبر',
    'c est vraiment bien ce projet bravo',
    'هاد الشي مقبولش بتاتا غلط كامل'
]

print('🧪 Live inference test:')
print('='*60)
for text in test_sentences:
    result = sentiment_pipeline(text, truncation=True, max_length=128)[0]
    label = result['label']
    score = result['score']
    emoji = {'positive': '😊', 'negative': '😞', 'neutral': '😐'}.get(label, '❓')
    print(f'{emoji} [{label.upper():8s}] {score:.2%} | {text}')

print('\n✅ Your Darija Sentiment Analyzer is ready!')